# Lecture 9 — GRPO: Group Relative Policy Optimization

> **CS40008.01: NLP & LLMs · Spring 2026 · Fudan University**

This notebook implements **GRPO** from scratch using HuggingFace Transformers and trains a
Qwen2.5-3B-Instruct model to solve the CountDown arithmetic task — the same setup as
[DeepSeek-R1-Zero](https://arxiv.org/abs/2501.12948).

**What you will learn:**
1. Why standard PPO is expensive for LLM fine-tuning, and how GRPO removes the critic.
2. The three-phase GRPO loop: *rollout → normalize → update*.
3. How to implement a verifiable reward function for a reasoning task.
4. How to write the GRPO policy-gradient loss with HuggingFace Transformers.
5. How to run a full training loop with W&B logging and checkpoint saving.

**Reference:** [GRPO-Zero](https://github.com/policy-gradient/GRPO-Zero) (minimal reference implementation)

**Hardware required:** NVIDIA A6000 48 GB (or any GPU with ≥ 24 GB VRAM)

---
## 1 · Background: Why RL for LLMs?

Supervised pre-training teaches a model to imitate text. But for **reasoning** tasks (math,
code, logic), imitation is not enough: there are many wrong paths and only a few right ones,
and the model needs to learn to *search* effectively.

Reinforcement Learning lets us train from a **reward signal** — a function that says whether
the model's output is correct — without needing human-written step-by-step solutions.

### 1.1 · Standard PPO for LLMs

Proximal Policy Optimization (PPO) is the standard RL algorithm for LLM fine-tuning
(used in RLHF for ChatGPT, Claude, etc.). It requires:

| Component | Purpose | Cost |
|-----------|---------|------|
| **Actor** $\pi_\theta$ | The LLM being trained | 1× model |
| **Critic** $V_\phi$ | Estimates expected return (baseline) | 1× model |
| **Reference** $\pi_{\text{ref}}$ | KL penalty anchor | 1× model |

→ You need to hold **3 copies** of the model (or close to it) in GPU memory simultaneously.

### 1.2 · GRPO: Remove the Critic

GRPO (Shao et al., DeepSeekMath, 2024) **replaces the critic network** with a statistical
baseline computed from a *group* of sampled responses:

> For each prompt, generate $G$ responses. Use their mean reward as the baseline
> instead of a learned value function.

The R1-Zero variant (DeepSeek-R1, 2025) additionally **drops the KL penalty** and
**drops PPO clipping**, making it a clean policy-gradient method:

$$\boxed{J(\theta) = \frac{1}{N_{\text{tok}}} \sum_{i=1}^{G} \sum_{t=1}^{|o_i|}
\log \pi_\theta(o_{i,t} \mid q,\, o_{i,<t}) \cdot \hat{A}_i}$$

where the group-relative advantage is:

$$\hat{A}_i = \frac{r_i - \mu_{\mathbf{r}}}{\sigma_{\mathbf{r}} + \varepsilon}$$

**Result:** Only 1 copy of the model needed. Training is simple, memory-efficient,
and surprisingly effective at inducing chain-of-thought reasoning.

---
## 2 · The GRPO Algorithm

The full training loop repeats three phases:

```
for each iteration:
  ┌─ ROLLOUT ──────────────────────────────────────────────────────┐
  │  1. Sample N questions from the dataset.                       │
  │  2. For each question q, generate G responses from π_θ.        │
  │  3. Score each response with reward function R(q, o).          │
  └────────────────────────────────────────────────────────────────┘
  ┌─ NORMALIZE ────────────────────────────────────────────────────┐
  │  4. For each question group, compute group mean μ and std σ.   │
  │  5. Normalize: Â_i = (r_i − μ) / (σ + ε)                      │
  └────────────────────────────────────────────────────────────────┘
  ┌─ UPDATE ────────────────────────────────────────────────────────┐
  │  6. Tokenize all (prompt, response) pairs.                     │
  │  7. Forward pass → per-token log-probs.                        │
  │  8. Compute loss = −(1/N_tok) Σ log π_θ(o_t) · Â_i            │
  │  9. Backprop + gradient clip + optimizer step (single update). │
  └────────────────────────────────────────────────────────────────┘
```

### Key design choices in R1-Zero

| Choice | Why |
|--------|-----|
| **No value network** | Group mean reward is a sufficient baseline for sparse binary rewards |
| **No KL penalty** | Single gradient step per rollout limits policy drift naturally |
| **No PPO clipping** | Same reason — one update per batch doesn't require trust-region protection |
| **Token-level loss** | Every token contributes equally; prevents short responses from dominating |
| **Verifiable reward** | Binary correct/incorrect — no reward model needed, no reward hacking |

---
## 3 · Setup

In [ ]:
import os
import re
import math
import random
import warnings
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import wandb
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU:  {gpu.name}')
    print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')

---
## 4 · Download the Model

We use **Qwen2.5-3B-Instruct** — a 3-billion-parameter instruction-tuned model from Alibaba.
In bfloat16 it uses ~6 GB of VRAM, leaving plenty of headroom on the A6000.

The download (~6 GB) will be cached locally so it only runs once.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MODEL_PATH = Path('./models/Qwen2.5-3B-Instruct')

if not MODEL_PATH.exists():
    print(f'Downloading {MODEL_NAME} ...')
    snapshot_download(
        repo_id=MODEL_NAME,
        local_dir=str(MODEL_PATH),
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*'],
    )
    print('Download complete!')
else:
    print(f'Model already at {MODEL_PATH}')

---
## 5 · The CountDown Task

CountDown is a simple arithmetic puzzle:

> Given a set of 3–4 numbers and a target, find an arithmetic expression using
> **each number exactly once** (with +, −, ×, ÷) that equals the target.

**Example:** numbers = `[1, 2, 3, 4]`, target = `11`  
**Valid answer:** `1 + (2 * 3) + 4 = 11` ✓

Why is this a good GRPO task?
- The reward is **binary and verifiable** — no learned reward model needed.
- Within each group of $G$ responses, some will be right and some wrong, creating
  a useful spread of rewards for the group-relative advantage.
- The task benefits from explicit chain-of-thought (the `<think>` block), so the model
  naturally learns to reason step by step.

In [ ]:
from datasets import load_dataset

raw = load_dataset('Jiayi-Pan/Countdown-Tasks-3to4', split='train')

TEST_SIZE = 128
test_ds  = raw.select(range(TEST_SIZE))
train_ds = raw.select(range(TEST_SIZE, len(raw)))

print(f'Train: {len(train_ds):,}  |  Test: {len(test_ds):,}')
print('\nSample problems:')
for ex in train_ds.select(range(5)):
    print(f'  nums={ex["nums"]}  target={ex["target"]}')

---
## 6 · Reward Functions

The reward has two components:

$$r = 0.1 \times r_{\text{format}} + r_{\text{answer}}$$

| Component | Value | Condition |
|-----------|-------|----------|
| $r_{\text{format}}$ | 1.0 | Response contains `<think>...</think>` **and** `<answer>...</answer>` |
| $r_{\text{format}}$ | 0.5 | Response has `<answer>` only |
| $r_{\text{format}}$ | 0.1 | Response has `<think>` only |
| $r_{\text{format}}$ | 0.0 | Neither tag present |
| $r_{\text{answer}}$ | 1.0 | Expression uses each number **exactly once** and evaluates to target |
| $r_{\text{answer}}$ | 0.0 | Otherwise |

The small format bonus (max 0.1) gently encourages the model to adopt the desired
output structure before it learns to solve problems correctly.

In [ ]:
SAFE_CHARS = set('0123456789 +-*/().')

def format_reward(text: str) -> float:
    """Reward for <think>...</think> + <answer>...</answer> format."""
    has_think  = bool(re.search(r'<think>.*?</think>',   text, re.DOTALL))
    has_answer = bool(re.search(r'<answer>.*?</answer>', text, re.DOTALL))
    if has_think and has_answer:
        return 1.0
    if has_answer:
        return 0.5
    if has_think:
        return 0.1
    return 0.0


def answer_reward(text: str, nums: List[int], target: int) -> float:
    """Reward for a correct arithmetic expression."""
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if not m:
        return 0.0
    expr = m.group(1).strip()
    # Only allow safe arithmetic characters to prevent code injection via eval
    if not all(c in SAFE_CHARS for c in expr):
        return 0.0
    # Each number from `nums` must appear exactly once
    found    = sorted(re.findall(r'\d+', expr))
    expected = sorted(str(n) for n in nums)
    if found != expected:
        return 0.0
    try:
        return 1.0 if abs(eval(expr) - target) < 1e-6 else 0.0
    except Exception:
        return 0.0


def compute_reward(text: str, nums: List[int], target: int) -> float:
    return 0.1 * format_reward(text) + answer_reward(text, nums, target)

In [ ]:
# Verify reward function on hand-crafted examples
test_cases = [
    ('<think>1+(2*3)+4=11</think>\n<answer>1 + (2 * 3) + 4</answer>', [1,2,3,4], 11, 'correct + full format'),
    ('<think>reasoning</think>\n<answer>1 + 2 + 3</answer>',          [1,2,3,4], 11, 'correct format, wrong answer (missing 4)'),
    ('<answer>1 + (2 * 3) + 4</answer>',                              [1,2,3,4], 11, 'correct answer, no <think>'),
    ('I think the answer is 11',                                       [1,2,3,4], 11, 'no tags at all'),
]

print(f'{"Description":<42} {"fmt":>5} {"ans":>5} {"total":>7}')
print('-' * 62)
for text, nums, target, desc in test_cases:
    fmt = format_reward(text)
    ans = answer_reward(text, nums, target)
    tot = compute_reward(text, nums, target)
    print(f'{desc:<42} {fmt:>5.1f} {ans:>5.1f} {tot:>7.2f}')

---
## 7 · Model & Tokenizer

We load **Qwen2.5-3B-Instruct** with:
- `torch_dtype=bfloat16` — halves memory vs float32, numerically stable on Ampere+ GPUs
- `device_map='auto'` — HuggingFace distributes layers across available GPUs automatically

We set `padding_side='left'` for the tokenizer because the model generates from the **right**
end of the sequence; left-padding keeps the real tokens together at the right.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading model (bfloat16) ...')
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH),
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params/1e9:.2f}B')
print(f'VRAM used:  {torch.cuda.memory_allocated()/1e9:.2f} GB')

---
## 8 · Prompt Formatting

Qwen2.5-Instruct uses a chat template. We apply it via `apply_chat_template`,
which inserts the correct special tokens (system/user/assistant delimiters).

The system prompt tells the model:
1. What the task is.
2. Exactly what format to use (`<think>` + `<answer>` tags).

This format is what the **format reward** checks — the model will be pushed to adopt it.

In [ ]:
SYSTEM_PROMPT = (
    'You are a mathematical reasoning assistant. '
    'Solve the CountDown problem step by step.\n\n'
    'Given a list of numbers and a target value, find an arithmetic expression '
    'using each number exactly once (with +, -, *, /) that equals the target.\n\n'
    'Always format your response as:\n'
    '<think>\n[step-by-step reasoning]\n</think>\n'
    '<answer>[arithmetic expression]</answer>'
)

def make_prompt(nums: List[int], target: int) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Numbers: {nums}\nTarget: {target}'},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

# Inspect a prompt
ex = train_ds[0]
prompt = make_prompt(ex['nums'], ex['target'])
n_toks = len(tokenizer.encode(prompt))
print(f'Problem: nums={ex["nums"]}  target={ex["target"]}')
print(f'Prompt length: {n_toks} tokens\n')
print(prompt)

### Sanity check: generation before training

Let's see what the untrained model produces. We expect poor answers (maybe the right format
after instruction tuning, but almost certainly wrong arithmetic).

In [ ]:
ex = train_ds[0]
prompt = make_prompt(ex['nums'], ex['target'])
inputs = tokenizer(
    prompt, return_tensors='pt', truncation=True, max_length=512
).to(device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

pl       = inputs['input_ids'].shape[1]
response = tokenizer.decode(out[0, pl:], skip_special_tokens=True)
reward   = compute_reward(response, ex['nums'], ex['target'])

print(f'nums={ex["nums"]}  target={ex["target"]}  →  reward={reward:.2f}\n')
print(response)

---
## 9 · Phase 1 — Rollout: Generating Response Groups

For each prompt $q$, we generate $G$ independent responses:

$$\{o_1, o_2, \ldots, o_G\} \sim \pi_\theta(\cdot \mid q)$$

HuggingFace's `model.generate(..., num_return_sequences=G)` does this in one call.
We then decode only the **generated part** (stripping the prompt tokens from the front).

We iterate over prompts one at a time (rather than batching all N prompts together)
to keep the implementation simple and avoid padding overhead for variable-length prompts.

In [ ]:
@torch.no_grad()
def generate_responses(
    model,
    tokenizer,
    prompts: List[str],
    num_responses: int,
    max_prompt_len: int,
    max_new_tokens: int,
    temperature: float = 1.0,
) -> List[List[str]]:
    """
    For each of N prompts, generate G independent responses.

    Returns: List[N] of List[G] response strings (prompt text NOT included).
    """
    model.eval()
    all_responses: List[List[str]] = []

    for prompt in prompts:
        inputs = tokenizer(
            prompt,
            return_tensors='pt',
            truncation=True,
            max_length=max_prompt_len,
            padding=False,
        ).to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_return_sequences=num_responses,
            do_sample=True,
            temperature=temperature,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )  # [G, prompt_len + response_len]

        # Decode only the new tokens (skip the prompt prefix)
        prompt_len = inputs['input_ids'].shape[1]
        responses  = tokenizer.batch_decode(
            outputs[:, prompt_len:], skip_special_tokens=True
        )
        all_responses.append(responses)

    return all_responses  # [N][G]

In [ ]:
# Quick demo: generate 4 responses for a single problem
ex = train_ds[42]
demo_responses = generate_responses(
    model, tokenizer,
    prompts=[make_prompt(ex['nums'], ex['target'])],
    num_responses=4,
    max_prompt_len=256,
    max_new_tokens=128,
)[0]

print(f'Problem: nums={ex["nums"]}  target={ex["target"]}\n')
for i, resp in enumerate(demo_responses):
    r = compute_reward(resp, ex['nums'], ex['target'])
    print(f'--- Response {i+1}  (reward={r:.2f}) ---')
    print(resp[:200], '...' if len(resp) > 200 else '')
    print()

---
## 10 · Phase 2 — Normalize: Group-Relative Advantages

For each question group of $G$ rewards $\{r_1, \ldots, r_G\}$:

$$\hat{A}_i = \frac{r_i - \mu_{\mathbf{r}}}{\sigma_{\mathbf{r}} + \varepsilon}$$

**Intuition:**
- $\hat{A}_i > 0$: this response was **better than average** → increase its probability
- $\hat{A}_i < 0$: this response was **worse than average** → decrease its probability
- $\hat{A}_i \approx 0$: near-average → almost no update

**Why this works better than a global baseline:**
Different problems have different difficulty. If we compared all responses globally,
easy problems (all correct) and hard problems (all wrong) would both contribute near-zero
advantage. Within-group normalization gives a useful signal regardless of difficulty.

**Edge case:** When all $G$ responses have the same reward ($\sigma = 0$), all advantages
become 0 and the update is zero — the model correctly learns nothing from this group.

In [ ]:
def compute_group_advantages(
    rewards_per_group: List[List[float]],
) -> List[List[float]]:
    """
    Z-score normalize rewards within each question group.

    Input:  [N][G] list of raw rewards
    Output: [N][G] list of advantages
    """
    advantages_per_group: List[List[float]] = []
    for group_rewards in rewards_per_group:
        r   = np.array(group_rewards, dtype=np.float32)
        mu  = r.mean()
        sig = r.std()
        advantages = ((r - mu) / (sig + 1e-8)).tolist()
        advantages_per_group.append(advantages)
    return advantages_per_group

In [ ]:
# Walkthrough example
example_rewards = [[1.1, 0.0, 1.1, 0.0, 1.1, 0.0, 0.05, 0.05]]
example_advantages = compute_group_advantages(example_rewards)

print('Raw rewards:  ', [f'{r:.2f}' for r in example_rewards[0]])
print('Advantages:   ', [f'{a:.2f}' for a in example_advantages[0]])
print(f'Mean: {np.mean(example_rewards[0]):.3f}  Std: {np.std(example_rewards[0]):.3f}')

---
## 11 · Phase 3 — Update: The GRPO Policy Gradient Loss

### 11.1 · Loss derivation

We want to **maximize** the expected advantage-weighted log-probability:

$$J(\theta) = \frac{1}{N_{\text{tok}}} \sum_{i=1}^{G} \sum_{t=1}^{|o_i|}
\underbrace{\log \pi_\theta(o_{i,t} \mid q, o_{i,<t})}_{\text{per-token log-prob}}
\cdot \underbrace{\hat{A}_i}_{\text{same for all tokens in } o_i}$$

Since PyTorch minimizes, our loss is $\mathcal{L} = -J(\theta)$.

### 11.2 · Computing log-probabilities with HuggingFace

A causal LM predicts token $t+1$ from tokens $0\ldots t$. Its output logits at position $t$
give the distribution over the next token. So:

```
logits = model(input_ids).logits          # [B, T, V]
shift_logits = logits[:, :-1, :]          # position 0..T-2 predict tokens 1..T-1
shift_labels = input_ids[:, 1:]           # tokens 1..T-1
log_probs = log_softmax(shift_logits)     # [B, T-1, V]
token_lp  = log_probs.gather(-1, labels) # [B, T-1]  — pick the actual token's log-prob
```

We apply the **response mask** to zero out prompt tokens — we only train on what
the model generated, not on the input question.

In [ ]:
def grpo_loss(
    model,
    sequences: torch.Tensor,       # [B, T]  token ids (prompt + response, left-padded)
    response_masks: torch.Tensor,  # [B, T]  1.0 for response tokens, 0.0 for prompt/pad
    advantages: torch.Tensor,      # [B]     group-normalised advantage per sequence
    pad_token_id: int,
) -> torch.Tensor:
    """
    GRPO policy gradient loss (R1-Zero: no KL penalty, no PPO clipping).

    Every response token contributes equally (token-level normalisation).
    The same advantage value Â_i is broadcast to all tokens in response i.
    """
    attention_mask = (sequences != pad_token_id).long()

    outputs = model(input_ids=sequences, attention_mask=attention_mask)
    logits  = outputs.logits  # [B, T, V]

    # Shift for next-token prediction
    shift_logits = logits[:, :-1, :].contiguous()     # [B, T-1, V]
    shift_labels = sequences[:, 1:].contiguous()       # [B, T-1]
    shift_masks  = response_masks[:, 1:].contiguous()  # [B, T-1]

    # Per-token log-probabilities (numerically equivalent to -cross_entropy)
    log_probs = F.log_softmax(shift_logits, dim=-1)    # [B, T-1, V]
    token_lp  = log_probs.gather(
        dim=-1, index=shift_labels.unsqueeze(-1)
    ).squeeze(-1)                                       # [B, T-1]

    # Zero out prompt and padding positions
    token_lp = token_lp * shift_masks                  # [B, T-1]

    # Advantage-weighted objective, normalised by total response tokens
    n_tokens = shift_masks.sum().clamp(min=1.0)
    obj = (token_lp * advantages.unsqueeze(1)).sum() / n_tokens
    return -obj  # negate: gradient descent on loss = gradient ascent on J

### 11.3 · Tokenizing episodes for training

For the backward pass, we need each (prompt, response) pair as a single token sequence
with a mask indicating which positions are response tokens.

We left-pad to handle variable-length sequences in each micro-batch.

In [ ]:
def prepare_batch(
    prompts:    List[str],
    responses:  List[str],
    advantages: List[float],
    tokenizer,
    max_prompt_len: int,
    max_response_len: int,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Tokenise (prompt, response) pairs and build response masks.

    Returns:
        sequences      [B, T]  left-padded integer tensor
        response_masks [B, T]  float tensor (1.0 = response token)
        advantages     [B]     float tensor
    """
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    all_ids, all_masks = [], []

    for prompt, response in zip(prompts, responses):
        p_ids = tokenizer.encode(prompt,   add_special_tokens=False)
        r_ids = tokenizer.encode(response, add_special_tokens=False)

        p_ids = p_ids[-max_prompt_len:]    # keep the most recent prompt tokens if too long
        r_ids = r_ids[:max_response_len]

        full_ids = p_ids + r_ids
        mask     = [0.0] * len(p_ids) + [1.0] * len(r_ids)
        all_ids.append(full_ids)
        all_masks.append(mask)

    # Left-pad all sequences to the same length
    max_len = max(len(s) for s in all_ids)
    padded_ids, padded_masks = [], []
    for ids, mask in zip(all_ids, all_masks):
        pad_len = max_len - len(ids)
        padded_ids.append([pad_id] * pad_len + ids)
        padded_masks.append([0.0]  * pad_len + mask)

    seq_t = torch.tensor(padded_ids,   dtype=torch.long)
    msk_t = torch.tensor(padded_masks, dtype=torch.float)
    adv_t = torch.tensor(advantages,   dtype=torch.float)
    return seq_t, msk_t, adv_t

---
## 12 · Training Configuration

Key parameters and their role:

| Parameter | Value | Why |
|-----------|-------|-----|
| `num_questions` | 32 | N: questions per iteration |
| `num_responses` | 8 | G: responses per question (256 total per iter) |
| `micro_batch_size` | 2 | Forward pass at a time; 128 accumulation steps per iter |
| `learning_rate` | 1e-5 | Conservative; policy changes should be small per step |
| `max_grad_norm` | 1.0 | Gradient clipping for stability |
| `temperature` | 1.0 | Full entropy sampling ensures response diversity |
| `num_iterations` | 1000 | ~3–5 hours on A6000 |

In [ ]:
config = dict(
    # ── Model ──────────────────────────────────
    model_path        = str(MODEL_PATH),
    max_prompt_len    = 256,
    max_response_len  = 1024,

    # ── Rollout ────────────────────────────────
    num_questions     = 32,     # N per iteration
    num_responses     = 8,      # G per question
    temperature       = 1.0,

    # ── Optimisation ───────────────────────────
    learning_rate     = 1e-5,
    weight_decay      = 0.01,
    max_grad_norm     = 1.0,
    micro_batch_size  = 2,      # sequences per gradient-accumulation micro-step
    num_iterations    = 1000,

    # ── Logging & checkpointing ─────────────────
    log_every         = 10,
    eval_every        = 50,
    save_every        = 100,
    save_dir          = './checkpoints',

    # ── W&B ────────────────────────────────────
    wandb_project     = 'grpo-countdown',
    wandb_run_name    = 'qwen2.5-3b-grpo',
)

---
## 13 · Evaluation

We evaluate with **greedy decoding** (temperature=0) on held-out problems.
Greedy evaluation is deterministic and gives a stable upper bound on performance.

In [ ]:
@torch.no_grad()
def evaluate(
    model, tokenizer, dataset, config: dict, n_samples: int = 64
) -> dict:
    """Greedy-decode on n_samples from dataset, return mean reward and solve rate."""
    model.eval()
    samples = random.sample(list(dataset), min(n_samples, len(dataset)))
    rewards, correct = [], []

    for s in samples:
        prompt = make_prompt(s['nums'], s['target'])
        inputs = tokenizer(
            prompt, return_tensors='pt',
            truncation=True, max_length=config['max_prompt_len'],
        ).to(device)
        out = model.generate(
            **inputs,
            max_new_tokens=config['max_response_len'],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        pl       = inputs['input_ids'].shape[1]
        response = tokenizer.decode(out[0, pl:], skip_special_tokens=True)
        r        = compute_reward(response, s['nums'], s['target'])
        rewards.append(r)
        correct.append(1.0 if r >= 1.0 else 0.0)

    return {
        'mean_reward': float(np.mean(rewards)),
        'solve_rate':  float(np.mean(correct)),
    }

---
## 14 · The Full GRPO Training Loop

Putting it all together: at each iteration we run all three phases, then log to W&B.

**Gradient accumulation:** We process 256 sequences (32×8) in micro-batches of 2,
accumulating gradients across 128 micro-steps before one optimizer step.
This lets us handle large batches without running out of VRAM.

**Sorting by length:** Before the gradient update we sort episodes by response length.
This groups similarly-sized sequences in each micro-batch, minimising padding waste.

**`torch.cuda.empty_cache()`:** We clear the GPU cache between generation and training
to avoid fragmentation from the large generation buffers.

In [ ]:
def train_grpo(model, tokenizer, train_ds, test_ds, config: dict) -> dict:
    """Run the full GRPO training loop. Returns a history dict for plotting."""
    Path(config['save_dir']).mkdir(parents=True, exist_ok=True)
    train_data = list(train_ds)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay'],
    )

    wandb.init(
        project=config['wandb_project'],
        name=config['wandb_run_name'],
        config=config,
        resume='allow',
    )

    history = {'reward': [], 'solve_rate': [], 'loss': []}

    for step in tqdm(range(config['num_iterations']), desc='GRPO'):

        # ── 1. Sample N questions ─────────────────────────────────────────
        questions = random.sample(train_data, config['num_questions'])
        prompts   = [make_prompt(q['nums'], q['target']) for q in questions]

        # ── 2. Generate G responses per question ──────────────────────────
        model.eval()
        torch.cuda.empty_cache()
        responses_per_q = generate_responses(
            model, tokenizer, prompts,
            num_responses  = config['num_responses'],
            max_prompt_len = config['max_prompt_len'],
            max_new_tokens = config['max_response_len'],
            temperature    = config['temperature'],
        )  # [N][G]

        # ── 3. Score every response ───────────────────────────────────────
        rewards_per_q = [
            [compute_reward(r, q['nums'], q['target']) for r in responses]
            for q, responses in zip(questions, responses_per_q)
        ]  # [N][G]

        # ── 4. Group-relative advantages ─────────────────────────────────
        adv_per_q = compute_group_advantages(rewards_per_q)  # [N][G]

        # ── 5. Flatten all N×G episodes ───────────────────────────────────
        flat_prompts, flat_responses, flat_advantages = [], [], []
        for prompt, responses, advantages in zip(prompts, responses_per_q, adv_per_q):
            flat_prompts.extend([prompt] * len(responses))
            flat_responses.extend(responses)
            flat_advantages.extend(advantages)

        # ── 6. Tokenise all episodes ──────────────────────────────────────
        sequences, resp_masks, adv_tensor = prepare_batch(
            flat_prompts, flat_responses, flat_advantages,
            tokenizer,
            max_prompt_len   = config['max_prompt_len'],
            max_response_len = config['max_response_len'],
        )
        # Sort by response length to minimise padding waste across micro-batches
        order      = resp_masks.sum(dim=1).long().argsort()
        sequences  = sequences[order]
        resp_masks = resp_masks[order]
        adv_tensor = adv_tensor[order]

        # ── 7. Gradient update with accumulation ──────────────────────────
        model.train()
        optimizer.zero_grad()

        n_seqs   = sequences.shape[0]
        n_micro  = math.ceil(n_seqs / config['micro_batch_size'])
        total_loss = 0.0

        for i in range(0, n_seqs, config['micro_batch_size']):
            mb_seq = sequences[i : i + config['micro_batch_size']].to(device)
            mb_msk = resp_masks[i : i + config['micro_batch_size']].to(device)
            mb_adv = adv_tensor[i : i + config['micro_batch_size']].to(device)

            loss = grpo_loss(
                model, mb_seq, mb_msk, mb_adv, tokenizer.pad_token_id
            )
            (loss / n_micro).backward()  # scale for accumulation
            total_loss += loss.item()

        torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
        optimizer.step()

        # ── 8. Metrics ────────────────────────────────────────────────────
        all_rewards = [r for group in rewards_per_q for r in group]
        mean_reward = float(np.mean(all_rewards))
        solve_rate  = float(np.mean([1.0 if r >= 1.0 else 0.0 for r in all_rewards]))
        avg_loss    = total_loss / n_micro

        history['reward'].append(mean_reward)
        history['solve_rate'].append(solve_rate)
        history['loss'].append(avg_loss)

        log_dict = {
            'train/mean_reward': mean_reward,
            'train/solve_rate':  solve_rate,
            'train/loss':        avg_loss,
        }

        # ── 9. Periodic evaluation ────────────────────────────────────────
        if (step + 1) % config['eval_every'] == 0:
            eval_m = evaluate(model, tokenizer, test_ds, config)
            log_dict.update({f'eval/{k}': v for k, v in eval_m.items()})
            tqdm.write(
                f'[{step+1:4d}] train reward={mean_reward:.4f} solve={solve_rate:.2%} '
                f'loss={avg_loss:.5f} | '
                f'eval reward={eval_m["mean_reward"]:.4f} solve={eval_m["solve_rate"]:.2%}'
            )
        elif (step + 1) % config['log_every'] == 0:
            tqdm.write(
                f'[{step+1:4d}] train reward={mean_reward:.4f} '
                f'solve={solve_rate:.2%} loss={avg_loss:.5f}'
            )

        wandb.log(log_dict, step=step)

        # ── 10. Checkpoint ────────────────────────────────────────────────
        if (step + 1) % config['save_every'] == 0:
            ckpt = Path(config['save_dir']) / f'step_{step+1:05d}'
            model.save_pretrained(str(ckpt))
            tokenizer.save_pretrained(str(ckpt))
            tqdm.write(f'  → checkpoint saved to {ckpt}')

    wandb.finish()
    return history

---
## 15 · Run Training

Before running, make sure you are logged in to W&B:
```bash
wandb login
```

Expected runtime on A6000 48 GB:
- Each iteration: ~20–30 s (generation dominates)
- 1000 iterations: ~5–8 hours

You can safely interrupt and re-run — the checkpoint cells below let you resume.

In [ ]:
history = train_grpo(model, tokenizer, train_ds, test_ds, config)

---
## 16 · Resume from Checkpoint (Optional)

If training was interrupted, load the latest checkpoint and continue.

In [ ]:
# Uncomment and adjust the path to resume from a checkpoint

# RESUME_PATH = './checkpoints/step_00500'

# print(f'Loading checkpoint from {RESUME_PATH} ...')
# model = AutoModelForCausalLM.from_pretrained(
#     RESUME_PATH,
#     torch_dtype=torch.bfloat16,
#     device_map='auto',
#     trust_remote_code=True,
# )
# tokenizer = AutoTokenizer.from_pretrained(RESUME_PATH, trust_remote_code=True)
# tokenizer.padding_side = 'left'
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
#
# history = train_grpo(model, tokenizer, train_ds, test_ds, config)

---
## 17 · Results

### 17.1 · Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['reward'])
axes[0].set_title('Mean Reward (train)')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Reward')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['solve_rate'])
axes[1].set_title('Solve Rate (train, sampled)')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Fraction solved')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['loss'])
axes[2].set_title('GRPO Loss')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Loss')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

### 17.2 · Final evaluation on the test set

In [ ]:
final_metrics = evaluate(model, tokenizer, test_ds, config, n_samples=128)
print(f'Final test mean reward: {final_metrics["mean_reward"]:.4f}')
print(f'Final test solve rate:  {final_metrics["solve_rate"]:.2%}')

### 17.3 · Qualitative examples

In [ ]:
model.eval()
print('Examples from trained model (greedy decoding):\n')

for i in range(8):
    s      = test_ds[i]
    prompt = make_prompt(s['nums'], s['target'])
    inputs = tokenizer(
        prompt, return_tensors='pt',
        truncation=True, max_length=config['max_prompt_len'],
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    pl       = inputs['input_ids'].shape[1]
    response = tokenizer.decode(out[0, pl:], skip_special_tokens=True)
    r        = compute_reward(response, s['nums'], s['target'])
    status   = 'SOLVED' if r >= 1.0 else 'wrong'

    print(f'{"="*65}')
    print(f'Problem : nums={s["nums"]}  target={s["target"]}')
    print(f'Reward  : {r:.2f}  [{status}]')
    print(f'Response:')
    print(response[:600])
    if len(response) > 600:
        print('  [... truncated ...]')
    print()

---
## 18 · Summary

### What happened during training

1. **Early iterations (0–100):** The model begins to adopt the `<think>/<answer>` format
   (pushed by the small format reward). Solve rate stays near 0.

2. **Middle phase (100–500):** Occasional correct answers appear. The group-relative
   advantage starts providing a clear signal — responses that happen to be correct
   get positive advantage; incorrect ones get negative. The policy learns to explore
   arithmetic more systematically in `<think>`.

3. **Late phase (500–1000):** Solve rate climbs steadily. The model develops consistent
   chain-of-thought patterns (trying combinations, checking results).

### Key takeaways

| Concept | Takeaway |
|---------|----------|
| **No critic needed** | Group mean is a sufficient baseline for binary verifiable rewards |
| **No reference model** | One update per rollout keeps the policy close enough |
| **Token-level loss** | Prevents short responses (less text to get wrong) from being unfairly advantaged |
| **Format reward** | Small signal that bootstraps the output structure before correctness signal kicks in |
| **Binary rewards** | Verifiability is key — no reward model, no reward hacking |

### Further reading

1. Shao et al. *DeepSeekMath* (2024) — original GRPO paper [`arXiv:2402.03300`](https://arxiv.org/abs/2402.03300)
2. DeepSeek-AI. *DeepSeek-R1* (2025) — R1-Zero removes KL and clipping [`arXiv:2501.12948`](https://arxiv.org/abs/2501.12948)
3. Yu et al. *DAPO* (2025) — token-level PG and removing KL at scale [`arXiv:2503.14476`](https://arxiv.org/abs/2503.14476)
4. GRPO-Zero — minimal reference implementation [`github.com/policy-gradient/GRPO-Zero`](https://github.com/policy-gradient/GRPO-Zero)